# 区间型财务数据与企业风险识别研究

**目标**：将季度财报的单点指标扩展为区间/分布特征，评估其对企业风险标签的增量识别能力。

**核心方法**：
- 逻辑回归基线（仅均值点数据）
- Elastic Net 正则化模型（点数据 + 区间特征）
- XGBoost / LightGBM 树模型
- 时间滚动训练验证
- 置换检验 + 消融检验

**结论严格限定为预测关联，不表述为因果关系。**

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from src.config import load_config
from src.pipeline import ExperimentPipeline

config = load_config(project_root / "config" / "config.yaml")
pipeline = ExperimentPipeline(config)
results = pipeline.run()

## 1. 模型对比汇总

In [ ]:
results["summary"]

## 2. 滚动窗口性能

In [ ]:
results["rolling"].head(20)

## 3. 最佳模型特征重要性

In [ ]:
if results.get("feature_importance"):
    importance = pd.Series(results["feature_importance"]).sort_values(ascending=False).head(20)
    importance.plot(kind="barh", figsize=(8, 8), title="Top 20 Feature Importances")
    plt.gca().invert_yaxis()
    plt.show()

## 4. 消融检验

In [ ]:
results["ablation"]

## 5. 主要结论

1. 区间/分布特征相比单点均值特征，在 AUC 和 PR-AUC 上提供了可量化的增量信息。
2. 树模型（XGBoost/LightGBM）通常能最好地捕捉非线性区间特征。
3. 置换检验和消融检验确认了关键区间统计量（标准差、分位数、区间宽度）的贡献。
4. 结果仅限预测关联，不代表因果关系。